In [38]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm
import random
import json
import langid
import sys
import regex as re  # Required for Unicode pattern matching
from typing import Generator, Optional, Set

files_processed_to_text = True




In [12]:
from datetime import datetime
def showTime():
    return str("["+datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')+" UTC]")

In [13]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [14]:
def decompress_zst_to_text(
    input_file: str,
    vocab: Optional[Set[str]] = None,
    mode: str = "accuracy",
    ascii_threshold: float = 0.5
) -> Generator[str, None, None]:
    """
    Decompresses a .zst file containing JSONL (JSON lines) format, 
    and yields English texts filtered via language detection or ASCII checks.
    
    ### Parameters
    input_file (str):
        Path to the .zst file containing JSONL-formatted lines.
        
    vocab (Optional[Set[str]]):
        A vocabulary set to collect unique characters from valid English texts. Defaults to None.
        
    mode (str, default='accuracy'):
        - 'accuracy': Uses the `langid` library for precise English language detection.
        - 'speed': Uses an ASCII ratio check for faster filtering.
        
    ascii_threshold (float, default=0.5):
        Minimum ASCII character ratio (0.0-1.0) for mode='speed' to consider text as valid.
    
    ### Yields
    str:
        Filtered English text entries from the compressed file.
    """
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = ""
            while True:
                chunk = reader.read(16384).decode("utf-8", errors="replace")  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split("\n")
                current_line = lines.pop() if lines else ""  # Save partial line for next iteration

                # Process each line
                for line in lines:
                    # Skip empty lines after stripping
                    stripped_line = line.strip()
                    if not stripped_line:
                        continue
                    
                    try:
                        data = json.loads(stripped_line)
                        text = data.get("text", "").strip()
                    except (json.JSONDecodeError, KeyError):
                        continue  # Skip invalid JSON
                        
                    except Exception as e:
                        print(f"JSON Error: {e} on line: {line[:50]}...")
                        continue

                    if mode == "accuracy":      
                        # Check language (English)
                        try:
                            detected_lang, _ = langid.classify(text)
                        except langid.langid.LanguageIdentificationError:
                            # Skip texts too short to identify
                            continue

                        if detected_lang != "en":
                            continue  # Non-English, skip

                    elif mode == "speed":
                        # ---- START FILTERING LOGIC ----
                        ascii_count = 0
                        total_chars = 0
                        
                        # Iterate through each character in text
                        for c in text:
                            code = ord(c)
                            if code <= 127:
                                ascii_count += 1
                            total_chars += 1

                        # Check filtering conditions
                        if total_chars == 0:
                            continue
                        if (ascii_count / total_chars) < ascii_threshold:
                            continue
                        # ---- END FILTERING LOGIC ----

                    # Update the vocabulary (only for English texts)
                    if vocab is not None:
                        vocab.update(set(text))

                    yield text.strip()


In [15]:
folder_path = "openwebtext2"
output_file = "output_v7_accuracy.txt"
vocab_file = "vocab_v7_accuracy.txt"


In [16]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")
print(files)
vocab = set()

Total files: 179
['2005-06.jsonl.zst', '2005-07.jsonl.zst', '2005-08.jsonl.zst', '2005-09.jsonl.zst', '2005-10.jsonl.zst', '2005-11.jsonl.zst', '2005-12.jsonl.zst', '2006-01.jsonl.zst', '2006-02.jsonl.zst', '2006-03.jsonl.zst', '2006-04.jsonl.zst', '2006-05.jsonl.zst', '2006-06.jsonl.zst', '2006-07.jsonl.zst', '2006-08.jsonl.zst', '2006-09.jsonl.zst', '2006-10.jsonl.zst', '2006-11.jsonl.zst', '2006-12.jsonl.zst', '2007-01.jsonl.zst', '2007-02.jsonl.zst', '2007-03.jsonl.zst', '2007-04.jsonl.zst', '2007-05.jsonl.zst', '2007-06.jsonl.zst', '2007-07.jsonl.zst', '2007-08.jsonl.zst', '2007-09.jsonl.zst', '2007-10.jsonl.zst', '2007-11.jsonl.zst', '2007-12.jsonl.zst', '2008-01.jsonl.zst', '2008-02.jsonl.zst', '2008-03.jsonl.zst', '2008-04.jsonl.zst', '2008-05.jsonl.zst', '2008-06.jsonl.zst', '2008-07.jsonl.zst', '2008-08.jsonl.zst', '2008-09.jsonl.zst', '2008-10.jsonl.zst', '2008-11.jsonl.zst', '2008-12.jsonl.zst', '2009-01.jsonl.zst', '2009-02.jsonl.zst', '2009-03.jsonl.zst', '2009-04.jsonl.z

In [17]:
# Shuffle files randomly 
random.seed(42)  # Optional: Set seed for reproducibility
random.shuffle(files)  # Shuffle in-place
print(files)

['2018-02.jsonl.zst', '2009-11.jsonl.zst', '2006-09.jsonl.zst', '2008-12.jsonl.zst', '2006-07.jsonl.zst', '2008-06.jsonl.zst', '2010-06.jsonl.zst', '2015-08.jsonl.zst', '2010-07.jsonl.zst', '2007-01.jsonl.zst', '2010-11.jsonl.zst', '2007-12.jsonl.zst', '2018-05.jsonl.zst', '2005-08.jsonl.zst', '2016-08.jsonl.zst', '2016-09.jsonl.zst', '2015-06.jsonl.zst', '2018-09.jsonl.zst', '2019-07.jsonl.zst', '2010-12.jsonl.zst', '2013-04.jsonl.zst', '2019-11.jsonl.zst', '2007-07.jsonl.zst', '2016-06.jsonl.zst', '2017-10.jsonl.zst', '2018-08.jsonl.zst', '2019-12.jsonl.zst', '2011-08.jsonl.zst', '2017-03.jsonl.zst', '2017-07.jsonl.zst', '2006-08.jsonl.zst', '2009-10.jsonl.zst', '2016-02.jsonl.zst', '2014-02.jsonl.zst', '2009-02.jsonl.zst', '2015-09.jsonl.zst', '2011-03.jsonl.zst', '2012-02.jsonl.zst', '2018-10.jsonl.zst', '2005-06.jsonl.zst', '2015-01.jsonl.zst', '2006-10.jsonl.zst', '2016-10.jsonl.zst', '2015-11.jsonl.zst', '2013-10.jsonl.zst', '2010-10.jsonl.zst', '2018-06.jsonl.zst', '2012-05.jso

In [18]:
# Process all files
if files_processed_to_text == False:
    with open(output_file, "w", encoding="utf-8") as outf:
        for filename in tqdm(files, total=len(files), desc="Processing Files"):
            print(f"{showTime()} Processing: {filename}")
            file_path = os.path.join(folder_path, filename)
            try:
                for text_line in decompress_zst_to_text(file_path, vocab, mode="accuracy"):
                    outf.write(text_line.strip())  # Write only the text line
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

In [19]:
# Write vocabulary
if files_processed_to_text == False:
    with open(vocab_file, "w", encoding="utf-8") as vfile:
        for char in sorted(vocab):
            vfile.write(char + "\n")

Corpus Preprocessing With Downsampling

In [30]:
input_path = output_file # corpus path
output_path_subset = "subset_v8_accuracy.txt"

In [31]:
def downsample_large_corpus(input_path: str, output_path_subset: str, target_size_gb: float = 2.0):
    """
    Randomly sample a subset of a large text file without loading into memory.

    Args:
        input_path (str): Path to the full dataset
        output_path_subset (str): Path to write the downsampled subset
        target_size_gb (float): Desired size in gigabytes (default 2GB)
    """
    
    # ----- Core Functionality -----
    
    # Calculate target size in bytes (1 gigabyte = 1e9 bytes)
    target_bytes = int(target_size_gb * 1e9)
    buffer = []  # Buffer to collect lines before writing (improves I/O efficiency)

    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(output_path_subset, 'w', encoding='utf-8') as outfile:
        
        # Step 1: Get the total number of lines in the entire file
        total_lines = 0
        for _ in infile:
            total_lines += 1
        # Reset file pointer to beginning after counting
        infile.seek(0)
        
        # Step 2: Calculate how many lines we need to reach target size
        bytes_per_line_avg = (os.fstat(infile.fileno()).st_size) / total_lines
        target_lines = int(target_bytes / bytes_per_line_avg)
        
        # Seed for reproducible sampling
        random.seed(42)

        # Step 3: Read and sample lines with progress tracking
        progress = tqdm(total=total_lines, desc="Reading")
        for line in infile:
            if random.random() < (target_lines / total_lines):
                # Add to buffer if selected
                buffer.append(line + '\n')
                # Write buffer to file when close to memory limit (0.1MB)
                if sys.getsizeof(buffer) > 0.1 * 1e6:
                    outfile.writelines(buffer)
                    buffer = []  # Reset buffer after writing
            progress.update(1)
        # Write remaining items in buffer after loop completes
        if buffer:
            outfile.writelines(buffer)

    print(f"Stored ~{target_size_gb}GB subset at {output_path_subset}")




In [ ]:
downsample_large_corpus(input_path, output_path_subset)

Reading: 100%|██████████| 546412743/546412743 [07:51<00:00, 1158338.80it/s]

Stored ~2.0GB subset at subset_v8_accuracy.txt


Custom Tokenizer

In [110]:
import re
import logging
from pathlib import Path
from typing import List, Optional, Pattern, Union

from tokenizers import Tokenizer, models, trainers, decoders
from tokenizers.normalizers import Sequence, NFKC, Lowercase
from tokenizers.pre_tokenizers import Split

# ------------------------------------------------------------------------------
# Module‐level logger
# ------------------------------------------------------------------------------
logger = logging.getLogger(__name__)
logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    level=logging.INFO
)

In [112]:
class HybridTokenizer:
    """
    A “hybrid” tokenizer combining:
      1) Regex‐based splitting for numbers & contractions
      2) Byte‐Pair Encoding (BPE) for subword segmentation

    Special tokens: [PAD], [UNK], [CLS], [SEP], [MASK]
    """

    def __init__(
        self,
        vocab_size: int = 30_000,
        min_frequency: int = 2,
        lowercase: bool = True,
        unicode_norm: bool = True,
        regex_pattern: Optional[Pattern[str]] = None,
    ):
        """
        Args:
            vocab_size: Desired vocabulary size for BPE.
            min_frequency: Minimum frequency threshold for BPE merges.
            lowercase: If True, apply lowercase normalization.
            unicode_norm: If True, apply NFKC unicode normalization.
            regex_pattern: Custom regex for pre‐tokenization. If None,
                           splits contractions & standalone numbers.
        """
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency

        # 1) Initialize BPE tokenizer with an [UNK] token
        self._tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

        # 2) Build normalization & pre‐tokenization
        self._setup_preprocessing(lowercase, unicode_norm, regex_pattern)

        # 3) Make sure decode() merges BPE fragments
        self._tokenizer.decoder = decoders.BPEDecoder()

        # 4) Define special tokens
        self.special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

    def _setup_preprocessing(
        self,
        lowercase: bool,
        unicode_norm: bool,
        pattern: Optional[Pattern[str]]
    ) -> None:
        """
        Configures the normalizer and attaches the built-in Split pre-tokenizer.
        """
        # A) Normalization (NFKC + lowercase)
        normalizers = []
        if unicode_norm:
            normalizers.append(NFKC())
        if lowercase:
            normalizers.append(Lowercase())
        if normalizers:
            self._tokenizer.normalizer = Sequence(normalizers)

        # B) Pre-tokenization: split out contractions & numbers
        if pattern is None:
            # default regex: contractions & standalone numbers
            regex_str = r"""
                (?i:[sdmt]|ll|ve|re) |       # 's, 'd, 'm, 't, 'll, 've, 're
                (?<!\S)\d+(?:\.\d+)*(?!\S)   # standalone integers or decimals
            """
        else:
            regex_str = pattern.pattern

        # Use the built-in Split with a string-pattern
        self._tokenizer.pre_tokenizer = Split(
            regex_str,
            behavior="isolated",  # the matches are emitted as their own tokens
            invert=False          # split on the pattern
        )

    def train(
        self,
        training_files: List[Union[str, Path]],
        output_dir: Union[str, Path]
    ) -> None:
        """
        Trains the BPE model on provided text corpora and saves artifacts.

        Args:
            training_files: List of text file paths to train on.
            output_dir: Directory to write 'vocab.json' & 'merges.txt'.
        """
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        # Ensure training files exist
        for f in training_files:
            if not Path(f).is_file():
                raise FileNotFoundError(f"Training file not found: {f}")

        logger.info(
            f"Training BPE (vocab_size={self.vocab_size}, "
            f"min_frequency={self.min_frequency})"
        )
        trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            min_frequency=self.min_frequency,
            special_tokens=self.special_tokens,
            show_progress=True
        )

        # Execute training
        self._tokenizer.train(
            files=[str(f) for f in training_files],
            trainer=trainer
        )

        # Persist vocab & merges
        self._tokenizer.model.save(
            str(output_dir / "vocab.json"),
            str(output_dir / "merges.txt")
        )
        logger.info(f"Saved vocab & merges to {output_dir.resolve()}")

    def encode(
        self,
        text: str,
        add_special_tokens: bool = True
    ) -> List[int]:
        """
        Encodes a string into token IDs.

        Args:
            text: Raw input string.
            add_special_tokens: If True, adds [CLS] at start and [SEP] at end.

        Returns:
            A list of integer token IDs.
        """
        encoding = self._tokenizer.encode(text, add_special_tokens=add_special_tokens)
        return encoding.ids

    def decode(
        self,
        token_ids: List[int],
        skip_special_tokens: bool = True
    ) -> str:
        """
        Decodes token IDs back into a string.

        Args:
            token_ids: Sequence of integer IDs.
            skip_special_tokens: If True, strips out special tokens.

        Returns:
            The reconstructed text.
        """
        return self._tokenizer.decode(token_ids, skip_special_tokens=skip_special_tokens)

    def save(self, output_dir: Union[str, Path]) -> None:
        """
        Saves the complete tokenizer to JSON for easy reload later.
        """
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        self._tokenizer.save(str(output_dir / "tokenizer.json"))
        logger.info(f"Tokenizer JSON saved to {output_dir/'tokenizer.json'}")

    @classmethod
    def from_file(cls, json_path: Union[str, Path]) -> "HybridTokenizer":
        """
        Reloads a tokenizer from a saved JSON file.

        Args:
            json_path: Path to 'tokenizer.json' produced by .save().
        """
        tok = Tokenizer.from_file(str(json_path))
        hybrid = cls.__new__(cls)
        hybrid._tokenizer = tok
        tok.decoder = decoders.BPEDecoder()
        # Original vocab_size/min_frequency aren’t preserved here
        hybrid.vocab_size = None
        hybrid.min_frequency = None
        hybrid.special_tokens = []
        return hybrid


Training Workflow

In [111]:
def train_tokenizer_workflow(
    training_files: List[Union[str, Path]],
    vocab_size: int = 30_000,
    output_dir: Union[str, Path] = "output_v8/en_hybrid_30k"
) -> HybridTokenizer:
    """
    End-to-end: create output directory, train, save, and sanity-check.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Output directory: {output_dir.resolve()}")

    tokenizer = HybridTokenizer(vocab_size=vocab_size)
    tokenizer.train(training_files, output_dir)

    # Sanity-check round-trip
    sample = "I'll test this tokenizer with numbers like 123.45 and emojis 😊!"
    ids = tokenizer.encode(sample)
    logger.info(f"Sample IDs: {ids}")
    logger.info(f"Round-trip: {tokenizer.decode(ids)!r}")

    tokenizer.save(output_dir)
    return tokenizer

In [109]:
pretrained = train_tokenizer_workflow(
    training_files=[output_path_subset],
    vocab_size=30000,
    output_dir="output_v8/en_hybrid_30k"
)

2025-05-02 19:42:13,648 [INFO] __main__: Output directory: F:\OneDrive\github\LLM_from_scratch\output_v8\en_hybrid_30k
2025-05-02 19:42:13,935 [INFO] __main__: Training BPE (vocab_size=30000, min_frequency=2)
